# Entrega 1

In [23]:
# Instalación de dependencias del proyecto
!pip install pandas matplotlib seaborn requests

In [24]:
# Importación de librerías y configuración global

import os
import requests
import pandas as pd
import os
import sys
import re

print("Librerías cargadas exitosamente y configuración inicial completa.")

Librerías cargadas exitosamente y configuración inicial completa.


In [25]:
def configurar_git():
  """Configura el entorno de Git según el host (Colab o Local)."""
  IN_COLAB = 'google.colab' in sys.modules
  if IN_COLAB:
    repo_url = "https://github.com/GomezMorales/TP_INTEGRADOR_CIENCIAS_DE_DATOS.git"
    repo_name = "TP_INTEGRADOR_CIENCIAS_DE_DATOS"
    if not os.path.exists(repo_name):
      !git clone -b Sprint_2 {repo_url}

    %cd {repo_name}

configurar_git()

Cloning into 'TP_INTEGRADOR_CIENCIAS_DE_DATOS'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 42 (delta 19), reused 23 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 805.19 KiB | 3.71 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/TP_INTEGRADOR_CIENCIAS_DE_DATOS/TP_INTEGRADOR_CIENCIAS_DE_DATOS


## Comprensión del negocio (Business Understanding -  CRISP-DM)

### Contexto del problema
El dataset pertenece al sector inmobiliario, específicamente a anuncios de propiedades publicadas en Zonaprop Argentina.
En el se incluye información típica del mercado: precios, ubicación, características físicas de las propiedades y atributos adicionales.

##### Relevancia
El mercado inmobiliario es un sector clave para compradores, vendedores, inversores, inmobiliarias y desarrolladores.
El análisis de datos permite entender tendencias de precios, demanda por zonas y características más valoradas,
aportando información útil para optimizar decisiones de inversión, estrategias comerciales y tasaciones.

##### Quién puede beneficiarse
* Inmobiliarias y agentes: para fijar precios adecuados y entender qué oferta tiene más demanda.
* Compradores/inversores: para identificar oportunidades y comparar zonas.
* Desarrolladores: para decidir dónde construir y qué tipo de vivienda ofrecer.
* Plataformas de venta/alquiler: para mejorar recomendaciones y segmentación de usuarios.

### Formulación del problema
¿Qué factores determinan el precio de una propiedad en el mercado
inmobiliario argentino, y cómo pueden los datos de Zonaprop ayudar
a entender y predecir esas variaciones?

### Objetivos del análisis (perspectiva del negocio)
* Identificar los principales determinantes del precio de una propiedad (por ejemplo, superficie, ubicación, cantidad de ambientes).
* Comprender cómo varían los precios entre diferentes barrios, ciudades o provincias.
* Detectar patrones que permitan caracterizar la oferta: tipos de propiedades más comunes, rangos de precio, características frecuentes.
* Apoyar decisiones estratégicas del negocio, como tasaciones, recomendaciones o inversión inmobiliaria.

### Preguntas analíticas que guiarán el trabajo
* ¿Cuáles son los factores más relacionados con el precio de una propiedad (superficie, ubicación, cantidad de ambientes, características adicionales)?
* ¿Cómo varían los precios promedio entre distintas zonas o ciudades dentro de Argentina?
* ¿Qué tipos de propiedades (casas, departamentos, PH, etc.) predominan en la oferta y cómo se diferencian en precio y características?

## Compresión de los datos (Data Understanding - CRISP-DM)

### Fuente de datos, obtención y condiciones de uso

El dataset fue obtenido del repositorio público de GitHub de Bright Data:
https://github.com/luminati-io/Zonaprop-Argentina-dataset-samples

Los datos fueron extraídos de Zonaprop Argentina mediante la API de
Bright Data (web scraping), y corresponden a anuncios de propiedades
inmobiliarias publicados en la plataforma.

El repositorio es público y de acceso libre. Bright Data ofrece acceso
gratuito a sus datasets para investigadores académicos y organizaciones
sin fines de lucro a través del programa Bright Initiative
(brightinitiative.com). El uso del dataset en este trabajo es con
fines estrictamente académicos.

In [26]:
# Carga del dataset
df = pd.read_csv('data/raw/Zonaprop Argentina - Properties Listing.csv')

# Estructura del dataset
print(f"Cantidad de filas: {df.shape[0]}")
print(f"Cantidad de columnas: {df.shape[1]}")

display(df.info())
display(df.head())

Cantidad de filas: 1000
Cantidad de columnas: 38
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 38 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   url                   1000 non-null   object 
 1   Title                 1000 non-null   object 
 2   generatedTitle        996 non-null    object 
 3   Imagenes              1000 non-null   object 
 4   Numero_de_imagenes    1000 non-null   int64  
 5   Description           1000 non-null   object 
 6   Precio                996 non-null    float64
 7   Currency              996 non-null    object 
 8   Fecha_de_publicacion  1000 non-null   object 
 9   Visualizaciones       484 non-null    float64
 10  Dimension_terreno     978 non-null    float64
 11  Dimension_propiedad   768 non-null    float64
 12  Ambientes             1000 non-null   int64  
 13  Banos                 1000 non-null   int64  
 14  Dormitorios           10

None

,url,Title,generatedTitle,Imagenes,Numero_de_imagenes,Description,Precio,Currency,Fecha_de_publicacion,Visualizaciones,...,Superdestacado,Premium_label,posting_id,proveedor_tour,estatus,latitude,longitude,address,seller_level,expenses
0,https://www.zonaprop.com.ar/propiedades/clasif...,Terreno en Venta - 366 m² - San Miguel del Monte,Terreno en Venta - 366 m² - San Miguel del Monte,"[""https://imgar.zonapropcdn.com/avisos/1/00/58...",14,_ Terreno en venta ubicado sobre calle Río Gua...,26000.0,USD,1/9/26,NaN,...,simple,Anunciante Premium,58076831,NaN,online,-35.436881,-58.820657,"Río Gualeguaychú y Cerro Aconcagua, San Miguel...",4.0,NaN
1,https://www.zonaprop.com.ar/propiedades/clasif...,Venta / Departamento 4 Ambientes / Macrocentro,Venta / Departamento 4 Ambientes / Macrocentro,"[""https://imgar.zonapropcdn.com/avisos/1/00/58...",22,Ofrecemos a la venta departamento 4 ambientes ...,110000.0,USD,1/6/26,NaN,...,destacado,Anunciante Premium,58040794,NaN,online,-37.995137,-57.557713,"20 de Septiembre al 1800, Macrocentro, Mar del...",4.0,NaN
2,https://www.zonaprop.com.ar/propiedades/clasif...,Venta Departamento 2 Amb. en Pozo Vista Al Mar,Venta Departamento 2 Amb. en Pozo Vista Al Mar,"[""https://imgar.zonapropcdn.com/avisos/1/00/58...",11,Corredor Responsable: Ariel Martin Simone REG ...,149000.0,USD,1/9/26,NaN,...,simple,Anunciante Premium,58067465,NaN,online,-38.059052,-57.544945,"Tripulantes del Fournier y Av, de los Trabajad...",3.0,NaN
3,https://www.zonaprop.com.ar/propiedades/clasif...,Venta Casa 4 Ambientes José León Suárez,Venta Casa 4 Ambientes José León Suárez,"[""https://imgar.zonapropcdn.com/avisos/1/00/58...",46,Corredor Responsable: GUILLERMO FRIMET CUCICBA...,120000.0,USD,1/8/26,NaN,...,simple,Anunciante Premium,58057978,NaN,online,-34.540526,-58.575771,"Sáenz Peña 3200, José León Suárez, General San...",4.0,NaN
4,https://www.zonaprop.com.ar/propiedades/clasif...,Venta Campo Productivo - Rivadavia - Mendoza,Venta Campo Productivo - Rivadavia - Mendoza,"[""https://imgar.zonapropcdn.com/avisos/1/00/58...",14,Corredor Responsable: Real Estate New Generati...,80000.0,USD,1/7/26,NaN,...,simple,Anunciante Premium,58048221,NaN,online,-33.201582,-68.465031,"Almirante Brown S/N, Rivadavia, Mendoza",3.0,NaN


In [27]:
df.describe()

,Numero_de_imagenes,Precio,Visualizaciones,Dimension_terreno,Dimension_propiedad,Ambientes,Banos,Dormitorios,Seller_ID,posting_id,latitude,longitude,seller_level,expenses
count,1000.000000,9.960000e+02,484.000000,9.780000e+02,768.000000,1000.000000,1000.000000,1000.000000,1.000000e+03,1.000000e+03,997.000000,997.000000,942.000000,2.710000e+02
mean,18.200000,3.079390e+05,77.946281,2.389698e+04,184.773438,2.568000,1.371000,1.657000,2.535051e+07,5.574173e+07,-34.416969,-59.660521,3.719745,2.251656e+05
std,11.472036,1.154490e+06,123.388147,4.800741e+05,541.980682,2.866635,1.457217,1.682909,6.392964e+06,3.251458e+06,2.984131,3.236820,0.529685,2.734423e+05
min,1.000000,0.000000e+00,15.000000,1.000000e+00,1.000000,0.000000,0.000000,0.000000,1.700946e+07,2.395502e+07,-54.502706,-80.119198,3.000000,1.000000e+00
25%,10.000000,6.300000e+04,22.000000,6.100000e+01,49.000000,0.000000,0.000000,0.000000,1.716108e+07,5.503454e+07,-34.809393,-59.150986,3.000000,7.000000e+04
50%,15.000000,1.192500e+05,37.000000,1.640000e+02,85.500000,3.000000,1.000000,2.000000,3.003904e+07,5.703541e+07,-34.601551,-58.528049,4.000000,1.500000e+05
75%,24.000000,2.503750e+05,81.000000,4.715000e+02,172.250000,4.000000,2.000000,3.000000,3.046537e+07,5.780627e+07,-34.388790,-58.380507,4.000000,3.000000e+05
max,50.000000,2.200000e+07,1481.000000,1.400000e+07,11111.000000,56.000000,18.000000,22.000000,3.076176e+07,5.808619e+07,25.990615,-54.399368,5.000000,3.192890e+06


### Análisis preliminar de calidad

In [28]:
# Porcentajes de datos faltantes por columna

# Fuente https://wesmckinney.com/book/data-cleaning
# Fuente https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.mean.html

naData = df.isna().mean() * 100

naData = naData[naData > 0].sort_values()

print("Porcentaje de datos faltantes por columna:")
print(naData)

Porcentaje de datos faltantes por columna:
Phone2                  0.1
latitude                0.3
longitude               0.3
generatedTitle          0.4
Operacion               0.4
Precio                  0.4
Currency                0.4
developmentFeatures     0.8
Dimension_terreno       2.2
seller_level            5.8
Nueva_usada            13.9
Phone1                 18.8
Dimension_propiedad    23.2
Zona3                  38.1
Visualizaciones        51.6
expenses               72.9
proveedor_tour         98.9
dtype: float64


In [29]:
# Detección de posibles inconsistencias e identificación de variables críticas para limpieza

print("Distribución de tipos de moneda:")
print(df['Currency'].value_counts())

superficie_cero = df[df['Dimension_propiedad'] == 0].shape[0]
print(f"Propiedades con superficie total igual a 0: {superficie_cero}")

inconsistencias = df[df['Dimension_propiedad'] > df['Dimension_terreno']]
print(f"Propiedades donde la sup. propiedad supera a la total: {inconsistencias.shape[0]}")

habitaciones_raras = df[df['Dormitorios'] >= df['Ambientes']]
print(f"Propiedades con más (o igual) dormitorios que ambientes totales: {habitaciones_raras.shape[0]}")

sin_banio = df[df['Banos'] == 0].shape[0]
print(f"Propiedades sin baños: {sin_banio}")

print("Distribución de tipos de inmueble:")
print(df['Tipo'].value_counts())


Distribución de tipos de moneda:
Currency
USD    918
ARS     78
Name: count, dtype: int64
Propiedades con superficie total igual a 0: 0
Propiedades donde la sup. propiedad supera a la total: 7
Propiedades con más (o igual) dormitorios que ambientes totales: 321
Propiedades sin baños: 254
Distribución de tipos de inmueble:
Tipo
Apartamento          374
Casa                 288
Terrenos             169
PH                    43
Local Comercial       35
Oficina comercial     30
Bodega-Galpon         15
Rancho                14
Garage                10
Edificio               6
Depósito               5
Hotel                  4
Vertical               3
Quinta Vacacional      2
Fondo de Comercio      1
Horizontal             1
Name: count, dtype: int64


In [30]:
## Valores extremos propiedades en USD
precios_usd = df[df['Currency'] == 'USD']['Precio']

q1 = precios_usd.quantile(0.25)
q3 = precios_usd.quantile(0.75)
iqr = q3 - q1

# Definimos límites para detectar outliers
limite_superior = q3 + 1.5 * iqr
limite_inferior = q1 - 1.5 * iqr

outliers_altos = precios_usd[precios_usd > limite_superior]
print(f"Propiedades con precio 'anormalmente' alto en USD: {outliers_altos.count()}")
print(outliers_altos.sort_values().head(10)) # Ver los 10 más bajos

# Identificar cuántos hay por debajo del límite
outliers_bajos = precios_usd[precios_usd < limite_inferior]
print(f"Propiedades con precio 'anormalmente' bajo en USD: {outliers_bajos.count()}")

Propiedades con precio 'anormalmente' alto en USD: 83
84     450000.0
82     450000.0
611    450000.0
448    450000.0
553    450000.0
506    450000.0
137    460000.0
149    465000.0
415    470000.0
820    470000.0
Name: Precio, dtype: float64
Propiedades con precio 'anormalmente' bajo en USD: 0


## Justificación de la elección del dataset

La elección del dataset proveniente de ZonaProp se fundamenta en su alta representatividad y riqueza multidimensional, factores críticos para un proyecto de Ciencia de Datos. Con una estructura de 38 variables (columnas) y una amplia cobertura geográfica en Argentina, el conjunto de datos ofrece la granularidad necesaria para aplicar técnicas avanzadas de análisis y modelado.

La idoneidad de esta fuente se vincula directamente con las preguntas analíticas de la siguiente manera:



*  Identificación de factores determinantes: La abundancia de variables tanto cuantitativas (superficie, ambientes, baños) como cualitativas (amenities, tipo de propiedad) permite realizar un análisis de correlación y selección de características (Feature Selection). Esto es fundamental para determinar estadísticamente qué factores tienen mayor impacto sobre el precio unitario.

*  Análisis geoespacial y comparativo: Dado que el dataset permite jerarquizar la información por provincias, localidades y barrios, es posible ejecutar análisis de segmentación y comparativa de medias. Esto garantiza una respuesta robusta sobre la variabilidad de precios entre distintas zonas de Argentina.

*  Caracterización de la oferta: El volumen de registros asegura una masa crítica de datos para cada categoría (Casas, Departamentos, PH), permitiendo realizar análisis descriptivos y de distribución que no se vean sesgados por la falta de muestras en segmentos específicos.

En conclusión, la riqueza de variables y la representatividad geográfica de
esta fuente aseguran que el dataset sea técnicamente apto para el desarrollo del proyecto. Su estructura facilita el procesamiento de datos necesario para desglosar la oferta inmobiliaria y comprender la dinámica de precios en sus distintas dimensiones.

# Entrega 2

### 4.1.A Preparación de los datos

In [31]:
df_clean = df.copy()

#### Eliminación de columnas

##### Eliminación de columnas que no aportan valor al dataset por el contenido de las mismas

In [32]:
"""
Justificación:

Se eliminan las columnas que poseen información a los servicios webs, referencian a atributos en una bbdd, o poseen información irrelevante.

Las columnas elegidas son:
 - url            = Url de la publicación, no posee relevancia estadística
 - generatedTitle = Derivado de Title con presencia de nulos, se opta por Title
 - Imagenes       = Url a las imagenes, no posee relevancia estadística
 - posting_id     = Referencia a un id en una tabla interna de la bbdd
 - Seller_name    = Sus valores se encuentran ofuscados, se opta por Seller_id
 - Seller_url     = Url del vendedor, sin relevancia estadistica
 - proveedor_tour = Url del logo del proveedor, sin relevancia estadística
 - developmentFeatures = Metadata de la web

 Columnas que poseen un solo valor asignado en la totalidad de registros.
  - Phone2        = Posee un solo valor o vacío, no aporta relevancia
  - Premium_label = Posee un solo valor o vacío, no aporta relevancia
  - estatus       = Posee un solo valor o vacío, no aporta relevancia"""


columnas_a_eliminar = [
    'url',
    'generatedTitle',
    'Imagenes',
    'posting_id',
    'Seller_name',
    'Seller_url',
    'proveedor_tour',
    'developmentFeatures',
    'Phone2',
    'Premium_label',
    'estatus'
]

df_clean = df_clean.drop(columns=columnas_a_eliminar)
print(f"Limpieza de columnas realizada. Columnas conservadas:\n{'\n'.join(df_clean.columns)}")

Limpieza de columnas realizada. Columnas conservadas:
Title
Numero_de_imagenes
Description
Precio
Currency
Fecha_de_publicacion
Visualizaciones
Dimension_terreno
Dimension_propiedad
Ambientes
Banos
Dormitorios
Tipo
Nueva_usada
Tipovendedor
Seller_ID
Phone1
Operacion
Zona
Zona2
Zona3
Superdestacado
latitude
longitude
address
seller_level
expenses


#### Normalización de cadenas

In [33]:
def clean_string_columns(df:pd.DataFrame) -> pd.DataFrame:
    """A todas las columnas de tipo Object le aplico minúsculas y quito espacios a izquierda y derecha, quito caracteres especiales a excepcion de "/" y "-" por los campos fecha."""

    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.strip().str.lower()
            # Eliminar caracteres especiales, manteniendo '/' y '-'
            df[col] = df[col].str.replace(r'[^a-z0-9\s/\-]', '', regex=True)
    return df

df_clean = clean_string_columns(df_clean)
print("Normalización de columnas de tipo Object finalizada.")

Normalización de columnas de tipo Object finalizada.


#### Casteos de columnas a tipos específicos

##### Casteo a string las columnas de tipo "Object"

In [34]:
def cast_to_string(df:pd.DataFrame,
                   columns_exception:list=[]) -> pd.DataFrame:
  """Realizo el casteo de las columnas tipo Object a string, de esta forma se
   mantiene la consistencia de los <NA> a lo largo de las columnas.
   Con el parámetro "columns_exception" podemos saltear el casteo  de una o
   varias columnas a string por motivos de conveniencia."""

  for col in df.columns:
      if (df[col].dtype == 'object') and (col not in columns_exception):
          df[col] = df[col].astype("string")

  return df

columns_exception = ['Fecha_de_publicacion']
df_clean = cast_to_string(df_clean,columns_exception=columns_exception)
print("Casteo a string finalizado.")

Casteo a string finalizado.


##### Casteo de int64 a Int64 y float64 a Float64

In [35]:
def cast_to_Int64_or_Float64(df:pd.DataFrame,
                             force_int:list=[],
                             force_float:list=[]) -> pd.DataFrame:
  """
  #Realizo el casteo de int64 a Int64 y float64 a Float64 para un mejor manejo
  # de los valores nulos y evitar comportamientos inesperados en calculos o
  # agrupamientos
  # Referencias:
  #  - https://pandas.pydata.org/docs/user_guide/integer_na.html
  #  - https://pandas.pydata.org/docs/reference/api/pandas.Float64Dtype.html

  #Los parámetros force_int y force_float es para forzar alguna columna específica
  # a ese formato.
  """
  for col in df.columns:
    if (df[col].dtype == 'int64') | (col in force_int):
        df[col] = df[col].astype('Int64')
    elif (df[col].dtype == 'float64') | (col in force_float):
          df[col] = df[col].astype('Float64')

  return df

force_to_int = ['Visualizaciones','seller_level']
df_clean = cast_to_Int64_or_Float64(df_clean, force_int=force_to_int)
print("Casteo a Int64 y Float64 finalizado.")

Casteo a Int64 y Float64 finalizado.


##### Castear a category columnas que tengan dicho comportamiento

In [36]:
def cast_as_category(df:pd.DataFrame,
                     columns_selected:list=[]) -> pd.DataFrame:
#A aquellas columnas que respondan al comportamiento de una categoría
# las transformo en ese tipo de dato para una mejor eficiencia en las consultas

  for col in columns_selected:
      df[col] = df[col].astype('category')
  return df

category_columns = [
    'Currency',
    'Tipo',
    'Nueva_usada',
    'Tipovendedor',
    'Operacion',
    'Zona',
    'Zona2',
    'Zona3',
    'Superdestacado'
]
df_clean = cast_as_category(df_clean,category_columns)

##### Castear a datetime campos fecha

In [37]:
def cast_to_datetime(df: pd.DataFrame,
                     columns:list=[],
                     date_format: str = '%d/%m/%Y') -> pd.DataFrame:
    """
    Castea una o varias columnas de un DataFrame a tipo datetime con un formato específico.

    Args:
        df (pd.DataFrame): El DataFrame de entrada.
        columns (str or list): El nombre de la columna o una lista de nombres de columnas a castear.
        date_format (str): El formato de fecha esperado (por defecto es '%d/%m/%Y').

    """
    for col in columns:
        if col in df.columns:
            # Usar errors='coerce' para convertir fechas no parseables en NaT (Not a Time)
            df[col] = pd.to_datetime(df[col], format=date_format, errors='coerce')

    return df

columns = ['Fecha_de_publicacion']
df_clean = cast_to_datetime(df_clean,columns=columns, date_format='%m/%d/%y')

In [38]:
numeric_columns = ['expenses']

#'Precio','Dimension_terreno','Dimension_propiedad',

all_outlier_indices = set()

for col in numeric_columns:
    print(f"\n--- Detección de Outliers para '{col}' ---")
    data_series = df_clean[col].dropna()

    if not data_series.empty:
        q1 = data_series.quantile(0.25)
        q3 = data_series.quantile(0.75)
        iqr = q3 - q1

        limite_superior = q3 + 1.5 * iqr
        limite_inferior = q1 - 1.5 * iqr

        outliers_altos = data_series[data_series > limite_superior]
        outliers_bajos = data_series[data_series < limite_inferior]

        print(f"Propiedades con '{col}' anormalmente alta: {outliers_altos.count()}")
        if not outliers_altos.empty:
            print("Valores (top 10):\n", outliers_altos.sort_values(ascending=False).head(10))
            all_outlier_indices.update(outliers_altos.index)

        print(f"Propiedades con '{col}' anormalmente baja: {outliers_bajos.count()}")
        if not outliers_bajos.empty:
            print("Valores (top 10):\n", outliers_bajos.sort_values(ascending=True).head(10))
            all_outlier_indices.update(outliers_bajos.index)
    else:
        print(f"La columna '{col}' no tiene valores numéricos para analizar después de eliminar nulos.")

print(f"\n---------------------------------------------------")
print(f"Total de registros únicos con al menos un outlier: {len(all_outlier_indices)}")


--- Detección de Outliers para 'expenses' ---
Propiedades con 'expenses' anormalmente alta: 14
Valores (top 10):
 408    3192890.0
883    1300000.0
868     900000.0
316     880000.0
391     850000.0
27      800000.0
474     800000.0
463     800000.0
308     730000.0
75      700000.0
Name: expenses, dtype: Float64
Propiedades con 'expenses' anormalmente baja: 0

---------------------------------------------------
Total de registros únicos con al menos un outlier: 14


### Eliminación de registros

#### Eliminación de registros con precio no definido

In [39]:
"""
Justificación:
  Se opta por eliminación del registro a aquellos donde el precio no este definido, ya que esta es una de las variables mas importantes del dataset y la muestra que cumple esta condición es pequeña.
"""

initial_rows = df_clean.shape[0]
df_clean.dropna(subset=['Precio'], inplace=True)
removed_rows = initial_rows - df_clean.shape[0]
print(f"Se eliminaron {removed_rows} registros donde 'Precio' era NA. El DataFrame ahora tiene {df_clean.shape[0]} filas.")

Se eliminaron 4 registros donde 'Precio' era NA. El DataFrame ahora tiene 996 filas.


In [40]:
df_clean['expenses'].describe()

,expenses
count,271.0
mean,225165.612546
std,273442.272153
min,1.0
25%,70000.0
50%,150000.0
75%,300000.0
max,3192890.0


#### Imputación de expenses

In [41]:
"""
Justificación:

 Debido a que se observo que una buena parte de los registros poseían <NA> o valores que se pueden asociar a un error de carga o un valor inválido en el campo 'expenses', se procedió a realizar el siguiente cálculo:
   Transformar estos valores a <NA>, crear una serie donde se quitan estos valores y luego calcular la mediana entre el Q1 y Q3 para obtener una mediana sin afectación de valores extremos.

 Posteriormente, imputar los valores <NA> en 'expenses' con esta mediana calculada. Esto permitirá conservar los registros y que estadísticamente no afecte en gran medida a la muestra.

 Se estableció como umbral mínimo de expensas a 10000, considerado un monto probablemente irreal en la actualidad.
"""

# Primero, identificamos y marcamos los valores implausiblemente bajos de 'expenses' como NaN.
# Por ejemplo, valores menores a 1000 (como 1.0, 8.0, 111.0) son probablemente errores o entradas incorrectas.
df_clean.loc[df_clean['expenses'] < 10000, 'expenses'] = pd.NA

# 1. Crear una serie con los valores no nulos de 'expenses' para el cálculo
expenses_valid_for_median = df_clean['expenses'].dropna().copy()

# Verificamos si hay suficientes valores para calcular una mediana
if not expenses_valid_for_median.empty:
    # 2. Calcular Q1 y Q3 sobre esta serie
    q1_expenses = expenses_valid_for_median.quantile(0.25)
    q3_expenses = expenses_valid_for_median.quantile(0.75)

    # 3. Filtrar los valores para incluir solo aquellos entre Q1 y Q3
    expenses_between_q1_q3 = expenses_valid_for_median[
        (expenses_valid_for_median >= q1_expenses) &
        (expenses_valid_for_median <= q3_expenses)
    ]

    # 4. Calcular la mediana de estos valores filtrados
    median_imputation_value = expenses_between_q1_q3.median()

    # Capturar la cantidad de registros NA antes de la imputación
    na_before_imputation = df_clean['expenses'].isna().sum()

    # 5. Imputar los valores NA en 'expenses' con esta mediana calculada
    df_clean['expenses'] = df_clean['expenses'].fillna(median_imputation_value)

    print(f"Q1 de expenses: {q1_expenses}")
    print(f"Q3 de expenses: {q3_expenses}")
    print(f"Mediana de expenses (entre Q1 y Q3) utilizada para imputación: {median_imputation_value}")
    print(f"Registros imputados en 'expenses' con la mediana: {na_before_imputation}")
else:
    print("No hay suficientes valores válidos en 'expenses' para calcular una mediana y realizar la imputación.")

print(f"Valores NA restantes en 'expenses' después de la imputación: {df_clean['expenses'].isna().sum()}")

Q1 de expenses: 85000.0
Q3 de expenses: 300000.0
Mediana de expenses (entre Q1 y Q3) utilizada para imputación: 160000.0
Registros imputados en 'expenses' con la mediana: 741
Valores NA restantes en 'expenses' después de la imputación: 0


#### Imputación Visualizaciones

In [ ]:
"""
Justificación:
 Aproximadamente el 51% de las observaciones sobre Visualizaciones estan informadas como <NA> esto puede deberse a un error en la recopilación de la métrica o bien omisión de registrar la misma. Esto traducido en un valor podría interpretarse como 0 visualizaciones para evitar eliminar la columna ya que puede resultar de importante para la muestra de datos.

 Para mantener un registro cual 0 es original de la observación y cual fue imputado se crea una columna nueva llamada 'Visualizaciones_fue_NA' donde sera valor True si ese 0 fue imputado y False si el 0 era original.
"""

In [44]:
# Crear la columna indicadora 'Visualizaciones_fue_NA' antes de imputar
df_clean['Visualizaciones_fue_NA'] = df_clean['Visualizaciones'].isna()

# Imputar los valores NA en 'Visualizaciones' con 0
initial_na_visualizaciones = df_clean['Visualizaciones'].isna().sum()
df_clean['Visualizaciones'] = df_clean['Visualizaciones'].fillna(0)

print(f"Se imputaron {initial_na_visualizaciones} valores NA en 'Visualizaciones' con 0.")
print(f"Se creó la columna 'Visualizaciones_fue_NA' para indicar los registros originalmente nulos.")
print(f"Valores NA restantes en 'Visualizaciones': {df_clean['Visualizaciones'].isna().sum()}")
display(df_clean[['Visualizaciones', 'Visualizaciones_fue_NA']].head())

Se imputaron 515 valores NA en 'Visualizaciones' con 0.
Se creó la columna 'Visualizaciones_fue_NA' para indicar los registros originalmente nulos.
Valores NA restantes en 'Visualizaciones': 0


,Visualizaciones,Visualizaciones_fue_NA
0,0,True
1,0,True
2,0,True
3,0,True
4,0,True


In [45]:
naData = df_clean.isna().mean() * 100

naData = naData[naData > 0].sort_values()

print("Porcentaje de datos faltantes por columna:")
print(naData)

Porcentaje de datos faltantes por columna:
longitude               0.301205
latitude                0.301205
Dimension_terreno       1.807229
seller_level            5.823293
Dimension_propiedad    22.891566
dtype: float64


In [43]:
df_clean['expenses'].describe()

,expenses
count,996.0
mean,180264.614458
std,143648.452807
min,10000.0
25%,160000.0
50%,160000.0
75%,160000.0
max,3192890.0


In [49]:
df_clean.head(5)

,Title,Numero_de_imagenes,Description,Precio,Currency,Fecha_de_publicacion,Visualizaciones,Dimension_terreno,Dimension_propiedad,Ambientes,...,Zona,Zona2,Zona3,Superdestacado,latitude,longitude,address,seller_level,expenses,Visualizaciones_fue_NA
0,terreno en venta - 366 m - san miguel del monte,14,terreno en venta ubicado sobre calle ro guale...,26000.0,usd,2026-01-09,0,366.0,<NA>,0,...,buenos aires fuera de gba,san miguel del monte,san miguel del monte,simple,-35.436881,-58.820657,ro gualeguaych y cerro aconcagua san miguel de...,4,160000.0,True
1,venta / departamento 4 ambientes / macrocentro,22,ofrecemos a la venta departamento 4 ambientes ...,110000.0,usd,2026-01-06,0,105.0,105.0,4,...,buenos aires costa atlntica,mar del plata,macrocentro,destacado,-37.995137,-57.557713,20 de septiembre al 1800 macrocentro mar del p...,4,160000.0,True
2,venta departamento 2 amb en pozo vista al mar,11,corredor responsable ariel martin simone reg 3...,149000.0,usd,2026-01-09,0,77.0,58.0,2,...,buenos aires costa atlntica,mar del plata,punta mogotes,simple,-38.059052,-57.544945,tripulantes del fournier y av de los trabajado...,3,160000.0,True
3,venta casa 4 ambientes jos len surez,46,corredor responsable guillermo frimet cucicba ...,120000.0,usd,2026-01-08,0,92.0,92.0,4,...,gba norte,general san martn,jos len surez,simple,-34.540526,-58.575771,senz pea 3200 jos len surez general san martn,4,160000.0,True
4,venta campo productivo - rivadavia - mendoza,14,corredor responsable real estate new generatio...,80000.0,usd,2026-01-07,0,16.0,<NA>,0,...,mendoza,rivadavia,nan,simple,-33.201582,-68.465031,almirante brown s/n rivadavia mendoza,3,160000.0,True
